# Frost Weather API: Oslo historical observations

Fetches 6-hourly weather (air_temperature, precipitation_amount, wind_speed, relative_humidity) for Oslo (Blindern SN18700) for all months that match the trip data range (April 2019 – January 2026). Uses `frost_fetch.py` for cache-before-fetch; only uncached months trigger API calls. Set `FORCE_WEATHER_FETCH=1` to bypass cache.

## Setup

In [1]:
# Paths, load .env from project root, get Frost client ID
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

cwd = Path.cwd()
project_root = cwd if (cwd / "package.json").exists() else cwd.parent.parent
env_path = project_root / ".env"
load_dotenv(env_path)

# Diagnose: so we can see if .env was found and what's set
print("Loading .env from:", env_path)
print(".env exists:", env_path.exists())
print("FROST_API_CLIENT_ID set:", bool(os.environ.get("FROST_API_CLIENT_ID")))
print("FROST_USER_AGENT set:", bool(os.environ.get("FROST_USER_AGENT")))

cache_dir = project_root / "weather-cache"

sys.path.insert(0, str(project_root / "data-pipeline"))
from frost_fetch import fetch_weather_month, fetch_weather_range

client_id = os.environ.get("FROST_API_CLIENT_ID")
if not client_id:
    raise RuntimeError(
        "Set FROST_API_CLIENT_ID in environment or .env. "
        "Copy .env.example to .env and add your Frost client ID."
    )

user_agent = os.environ.get("FROST_USER_AGENT", "").strip()
if not user_agent:
    raise RuntimeError(
        "FROST_USER_AGENT is missing. MET blocks requests without a User-Agent with contact info.\n\n"
        "1. Open the file shown above as 'Loading .env from' in your editor.\n"
        "2. Add exactly this line (use your real email, no quotes):\n"
        "   FROST_USER_AGENT=INF252-Course-Project/1.0 (your.email@example.com)\n"
        "3. Save the file and re-run this Setup cell.\n\n"
        "If .env exists but FROST_USER_AGENT still shows False, check the line is not commented with # and the variable name has no typo."
    )

# Show redacted User-Agent so you can confirm it was loaded
ua_redact = user_agent[:40] + "..." if len(user_agent) > 45 else user_agent
print("Using User-Agent:", ua_redact)
print("Project root:", project_root)
print("Cache dir:", cache_dir)

Loading .env from: /home/sage/Downloads/apps/INF252-Course-Project/.env
.env exists: True
FROST_API_CLIENT_ID set: True
FROST_USER_AGENT set: True
Using User-Agent: INF252-Course-Project/1.0 (nicolasmmjoes...
Project root: /home/sage/Downloads/apps/INF252-Course-Project
Cache dir: /home/sage/Downloads/apps/INF252-Course-Project/weather-cache


## Minimal request (debug 403)

If the single-month test below returns 403, run this cell: it sends a minimal request matching [Frost's official example](https://frost.met.no/python_example.html) (one element, 3 days). If this also returns 403, the problem is likely your **client ID** (get one from [frost.met.no/auth/requestCredentials.html](https://frost.met.no/auth/requestCredentials.html), use the exact string from the email). If this returns 200, the issue is with our full request parameters.

In [2]:
import requests
url = "https://frost.met.no/observations/v0.jsonld"
params = {"sources": "SN18700", "elements": "mean(air_temperature P1D)", "referencetime": "2010-04-01/2010-04-03"}
r = requests.get(url, params=params, auth=(client_id.strip(), ""), headers={"User-Agent": user_agent}, timeout=30)
print("Status:", r.status_code)
if r.status_code != 200:
    print("Response:", r.text[:600])
else:
    print("OK – minimal request works. Client ID and User-Agent are accepted.")

Status: 200
OK – minimal request works. Client ID and User-Agent are accepted.


## Test: fetch one month first

Run this before the full range to verify your client ID and User-Agent. If you get 403, add to `.env`: `FROST_USER_AGENT=INF252-Course-Project/1.0 (your.email@example.com)` — MET requires a User-Agent with contact info.

In [3]:
# Single-month test (e.g. Feb 2024)
test = fetch_weather_month(2024, 2, client_id, cache_dir)
print(f"OK: {test['year']}-{test['month']:02d}, cached={test['cached']}, data keys={list(test.get('response', {}).keys())}")

OK: 2024-02, cached=False, data keys=['@context', '@type', 'apiVersion', 'license', 'createdAt', 'queryTime', 'currentItemCount', 'itemsPerPage', 'offset', 'totalItemCount', 'currentLink', 'data']


## Fetch all months (2019-04 … 2026-01)

In [4]:
# Same range as trip data: April 2019 through January 2026
results = fetch_weather_range(
    2019, 4, 2026, 1,
    client_id,
    cache_dir,
)

cached_count = sum(1 for r in results if r.get("cached"))
fetched_count = len(results) - cached_count
print(f"Months: {len(results)} total, {fetched_count} fetched, {cached_count} from cache")

Months: 82 total, 23 fetched, 59 from cache


## Inspect one month (optional)

In [5]:
# Example: first month's response structure
r = results[0]
data = r.get("response", {}).get("data", [])
print(f"Source: {r['source_id']}, {r['year']}-{r['month']:02d}, cached={r['cached']}")
if data:
    print(f"Series count: {len(data)}")
    first = data[0]
    print(f"Keys: {list(first.keys())}")
    obs = first.get("observations", [])[:3]
    print(f"Sample observations: {obs}")

Source: SN18700, 2019-04, cached=True
Series count: 4305
Keys: ['sourceId', 'referenceTime', 'observations']
Sample observations: [{'elementId': 'air_temperature', 'value': -2, 'unit': 'degC', 'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2}, 'timeOffset': 'PT0H', 'timeResolution': 'PT1H', 'timeSeriesId': 0, 'performanceCategory': 'C', 'exposureCategory': '1', 'qualityCode': 0}, {'elementId': 'wind_speed', 'value': 0.9, 'unit': 'm/s', 'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 10}, 'timeOffset': 'PT0H', 'timeResolution': 'PT1H', 'timeSeriesId': 0, 'performanceCategory': 'C', 'exposureCategory': '2', 'qualityCode': 0}, {'elementId': 'wind_speed', 'value': 0.9, 'unit': 'm/s', 'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 10}, 'timeOffset': 'PT0H', 'timeResolution': 'PT10M', 'timeSeriesId': 0, 'performanceCategory': 'C', 'exposureCategory': '2', 'qualityCode': 0}]
